# Modelo–Vista–Controlador (MVC) — Sistema de procesamiento de pedidos

**Objetivo.** Comprender el patrón arquitectónico **Modelo–Vista–Controlador (MVC)** con un ejemplo sencillo en Python.

Trabajaremos con el mismo sistema transversal de la unidad: **procesamiento de pedidos**.

- **Vista:** presenta información y captura acciones.
- **Controlador:** interpreta eventos y coordina.
- **Modelo:** mantiene el estado y ejecuta las reglas de negocio.

## 1. El problema: responsabilidades mezcladas

Primero ejecutaremos una versión donde una sola función captura datos, aplica reglas y muestra resultados.

In [75]:
def registrar_pedido_mezclado():
    pedido = {
        "cliente": "Ana",
        "productos": [
            {"nombre": "Teclado", "precio": 500, "cantidad": 1},
            {"nombre": "Mouse", "precio": 250, "cantidad": 2}
        ]
    }

    if not pedido["productos"]:
        print("Error: el pedido no contiene productos")
        return

    subtotal = sum(
        p["precio"] * p["cantidad"]
        for p in pedido["productos"]
    )

    descuento = subtotal * 0.10 if subtotal >= 1000 else 0
    impuestos = round((subtotal - descuento) * 0.16, 2)
    total = round(subtotal - descuento + impuestos, 2)

    pedido["subtotal"] = subtotal
    pedido["descuento"] = descuento
    pedido["impuestos"] = impuestos
    pedido["total"] = total
    pedido["estado"] = "PROCESADO"

    print("Pedido registrado")
    print("Cliente:", pedido["cliente"])
    print("Total:", pedido["total"])
    print("Estado:", pedido["estado"])

registrar_pedido_mezclado()

Pedido registrado
Cliente: Ana
Total: 1044.0
Estado: PROCESADO


### Preguntas para discutir

1. ¿Qué responsabilidades diferentes aparecen en la función?
2. ¿Qué parte cambia si cambia la forma de mostrar el pedido?
3. ¿Qué parte cambia si cambia la regla de descuento?
4. ¿Qué problema aparece si la interfaz crece?

## 2. Separando responsabilidades con MVC

## 3. Modelo

El Modelo contiene el estado y las reglas del dominio. No debe encargarse de la interfaz.

In [76]:
class PedidoModelo:

    def registrar_pedido(self, pedido):
        if not pedido.get("cliente"):
            raise ValueError("Cliente requerido")

        if not pedido.get("productos"):
            raise ValueError("El pedido no contiene productos")

        for producto in pedido["productos"]:
            if producto["cantidad"] <= 0:
                raise ValueError("La cantidad debe ser mayor que cero")

        subtotal = sum(
            p["precio"] * p["cantidad"]
            for p in pedido["productos"]
        )

        descuento = subtotal * 0.10 if subtotal >= 1000 else 0
        impuestos = round((subtotal - descuento) * 0.16, 2)
        total = round(subtotal - descuento + impuestos, 2)

        pedido["subtotal"] = subtotal
        pedido["descuento"] = descuento
        pedido["impuestos"] = impuestos
        pedido["total"] = total
        pedido["estado"] = "PROCESADO"

        return pedido

## 4. Vista

La Vista captura/presenta información. No debe implementar reglas de negocio.

In [77]:
class PedidoVista:

    def capturar_pedido(self):
        return {
            "cliente": "Ana",
            "productos": [
                {"nombre": "Teclado", "precio": 500, "cantidad": 1},
                {"nombre": "Mouse", "precio": 250, "cantidad": 2}
            ]
        }

    def mostrar_resultado(self, pedido):
        print("Pedido registrado")
        print("Cliente:", pedido["cliente"])
        print("Subtotal:", pedido["subtotal"])
        print("Descuento:", pedido["descuento"])
        print("Impuestos:", pedido["impuestos"])
        print("Total:", pedido["total"])
        print("Estado:", pedido["estado"])

    def mostrar_error(self, mensaje):
        print("Error:", mensaje)

## 5. Controlador

El Controlador interpreta la acción y coordina Vista y Modelo. No debe calcular descuentos ni impuestos.

In [78]:
class PedidoControlador:

    def __init__(self, modelo, vista):
        self.modelo = modelo
        self.vista = vista

    def registrar_pedido(self):
        try:
            pedido = self.vista.capturar_pedido()
            resultado = self.modelo.registrar_pedido(pedido)
            self.vista.mostrar_resultado(resultado)

        except ValueError as error:
            self.vista.mostrar_error(str(error))

## 6. Ejecutemos MVC

In [79]:
modelo = PedidoModelo()
vista = PedidoVista()
controlador = PedidoControlador(modelo, vista)

controlador.registrar_pedido()

Pedido registrado
Cliente: Ana
Subtotal: 1000
Descuento: 100.0
Impuestos: 144.0
Total: 1044.0
Estado: PROCESADO


## 7. Sigamos el flujo

In [80]:
print("1. Usuario realiza una acción")
print("2. Vista captura los datos")
print("3. Controlador recibe el evento")
print("4. Controlador solicita la operación al Modelo")
print("5. Modelo valida y calcula")
print("6. Modelo devuelve el resultado")
print("7. Controlador solicita a la Vista mostrarlo")

1. Usuario realiza una acción
2. Vista captura los datos
3. Controlador recibe el evento
4. Controlador solicita la operación al Modelo
5. Modelo valida y calcula
6. Modelo devuelve el resultado
7. Controlador solicita a la Vista mostrarlo


## 8. Cambio de Vista

Cambiaremos únicamente la forma de presentar los resultados.

In [81]:
class PedidoVistaResumida(PedidoVista):

    def mostrar_resultado(self, pedido):
        print(
            f"Pedido {pedido['estado']} | "
            f"Total: ${pedido['total']:.2f}"
        )

In [82]:
vista_resumida = PedidoVistaResumida()
controlador_resumido = PedidoControlador(
    modelo,
    vista_resumida
)

controlador_resumido.registrar_pedido()

Pedido PROCESADO | Total: $1044.00


### Pregunta

¿Fue necesario modificar `PedidoModelo` para cambiar la presentación?

¿Qué atributo de calidad se favorece?

## 9. Detectemos una violación de MVC

Analiza:

```python
class PedidoControlador:

    def registrar_pedido(self):
        subtotal = sum(
            p["precio"] * p["cantidad"]
            for p in pedido["productos"]
        )

        descuento = subtotal * 0.10
```

Responde:

1. ¿Qué responsabilidad está asumiendo el Controlador?
2. ¿A qué componente debería pertenecer?
3. ¿Qué problema aparece si varios controladores duplican esas reglas?

## 10. Actividad: agregar consulta de pedidos

Completa los `TODO`.

In [83]:
class PedidoModeloConMemoria:

    def __init__(self):
        self.pedidos = {}
        self.siguiente_id = 1

    def registrar_pedido(self, pedido):
        if not pedido.get("cliente"):
            raise ValueError("Cliente requerido")

        if not pedido.get("productos"):
            raise ValueError("El pedido no contiene productos")

        subtotal = sum(
            p["precio"] * p["cantidad"]
            for p in pedido["productos"]
        )

        descuento = subtotal * 0.10 if subtotal >= 1000 else 0
        impuestos = round((subtotal - descuento) * 0.16, 2)
        total = round(subtotal - descuento + impuestos, 2)

        pedido["id"] = self.siguiente_id
        pedido["subtotal"] = subtotal
        pedido["descuento"] = descuento
        pedido["impuestos"] = impuestos
        pedido["total"] = total
        pedido["estado"] = "PROCESADO"

        self.pedidos[self.siguiente_id] = pedido.copy()
        self.siguiente_id += 1

        return pedido

    def consultar_pedido(self, id_pedido):
        # TODO: devolver el pedido si existe o None
        if(id_pedido < 1 or id_pedido >= self.siguiente_id):
            return None
        
        return self.pedidos[id_pedido]

In [84]:
class PedidoVistaConsulta(PedidoVista):

    def mostrar_pedido(self, pedido):
        print("Pedido encontrado")
        print("Cliente:", pedido["cliente"])
        print("Subtotal:", pedido["subtotal"])
        print("Descuento:", pedido["descuento"])
        print("Impuestos:", pedido["impuestos"])
        print("Total:", pedido["total"])
        print("Estado:", pedido["estado"])

    def mostrar_no_encontrado(self):
        super().mostrar_error("pedido no encontrado")
        pass

In [85]:
class PedidoControladorConsulta(PedidoControlador):

    def consultar_pedido(self, id_pedido):
        pedidoBuscado = modelo.consultar_pedido(id_pedido)
        if(pedidoBuscado == None):
            vista.mostrar_no_encontrado()
        else:
            vista.mostrar_pedido(pedidoBuscado)
        pass

## 11. Reto: segunda Vista, mismo Modelo

Implementa `PedidoVistaSoloTotal`.

Debe mostrar únicamente:

```text
Pedido 1
Total: $...
```

**Condición:** no modifiques el Modelo.

In [86]:
class PedidoVistaSoloTotal(PedidoVistaConsulta):

    def mostrar_pedido(self, pedido):
            print("pedido encontrado")
            print(f"total = {pedido["total"]}")
    
    def mostrar_no_encontrado(self):
        super().mostrar_error("pedido no encontrado")
        pass

### Pruebas

In [87]:
modelo = PedidoModelo()
vista = PedidoVista()
controlador = PedidoControlador(modelo, vista)

controlador.registrar_pedido()

Pedido registrado
Cliente: Ana
Subtotal: 1000
Descuento: 100.0
Impuestos: 144.0
Total: 1044.0
Estado: PROCESADO


In [88]:
modelo = PedidoModeloConMemoria()
vista = PedidoVistaConsulta()
controlador = PedidoControladorConsulta(modelo, vista)

controlador.registrar_pedido()

id_existente = 1
id_no_existente = 2

print("\nConsultando pedido Existente\n")

controlador.consultar_pedido(id_existente)

print("\nConsultando pedido No Existente\n")

controlador.consultar_pedido(id_no_existente)

Pedido registrado
Cliente: Ana
Subtotal: 1000
Descuento: 100.0
Impuestos: 144.0
Total: 1044.0
Estado: PROCESADO

Consultando pedido Existente

Pedido encontrado
Cliente: Ana
Subtotal: 1000
Descuento: 100.0
Impuestos: 144.0
Total: 1044.0
Estado: PROCESADO

Consultando pedido No Existente

Error: pedido no encontrado


In [89]:
print("Nueva vista")
vista = PedidoVistaSoloTotal()
controlador = PedidoControladorConsulta(modelo, vista)

controlador.consultar_pedido(id_existente)

Nueva vista
pedido encontrado
total = 1044.0


## 12. Reflexión final

1. ¿Qué responsabilidad tiene el Modelo?
2. ¿Qué responsabilidad tiene la Vista?
3. ¿Qué responsabilidad tiene el Controlador?
4. ¿Por qué el Controlador no debería contener reglas complejas de negocio?
5. ¿Qué beneficio se obtiene al usar varias vistas con el mismo Modelo?
6. ¿En qué tipo de aplicación MVC podría resultar excesivo?
7. ¿En qué se diferencia MVC de una arquitectura en capas?

## Idea clave

**MVC separa la interacción con el usuario, el control del flujo y el estado/comportamiento del sistema.**

La Vista presenta y captura; el Controlador interpreta y coordina; el Modelo mantiene el estado y ejecuta las reglas del dominio.